# Activity 1: RAGAS Evaluation with Cost Analysis

This notebook builds **two comparable RAG pipelines** over `data/cat-health-guide.pdf` —
one powered by open-source models on **Fireworks AI**, one powered by **OpenAI `gpt-4.1-mini`**
— and compares them on:

1. **Retrieval quality, faithfulness & end-to-end accuracy** using RAGAS (with a synthetic
   test set generated by `TestsetGenerator`)
2. **Token usage and cost per query**, instrumented via **LangSmith** tracing

Both pipelines share identical chunking, prompting, and retrieval logic — the only
difference is the embedding model and chat model — so the comparison isolates the effect
of the provider/model choice.


In [2]:
import os
import getpass

from dotenv import load_dotenv

load_dotenv()

for var in ["FIREWORKS_API_KEY", "OPENAI_API_KEY"]:
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"Enter your {var}: ")

# LangSmith tracing is optional but required for the cost/tracing portion of this activity.
if os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ.setdefault("LANGSMITH_PROJECT", "rag-ragas-eval")
    print("LangSmith tracing enabled.")
else:
    print(
        "LANGSMITH_API_KEY not set - tracing/cost sections will run but skip LangSmith. "
        "Add it to .env to enable full instrumentation."
    )


LangSmith tracing enabled.


## 1. Load & chunk the source document

Both pipelines are built from the *same* chunks, so retrieval differences are due to the embedding model, not the data.

In [3]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken

DATA_PATH = "data/cat-health-guide.pdf"


def _tiktoken_len(text: str) -> int:
    return len(tiktoken.encoding_for_model("gpt-4o").encode(text))


documents = PyMuPDFLoader(DATA_PATH).load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=750, chunk_overlap=0, length_function=_tiktoken_len
)
chunks = text_splitter.split_documents(documents)

print(f"Loaded {len(documents)} pages -> {len(chunks)} chunks")


Loaded 22 pages -> 42 chunks


## 2. Build two comparable RAG pipelines

A single `build_rag_pipeline` factory builds a retrieve-then-generate pipeline from
whatever embedding model + chat model it's given. Each pipeline gets its own in-memory
Qdrant collection built from the *same* `chunks`.

Each pipeline's `run()` returns the response, the retrieved contexts (for RAGAS), and the
generation's `usage_metadata` (for the cost analysis later).

In [9]:
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

RAG_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "human",
            "\n#CONTEXT:\n{context}\n\nQUERY:\n{query}\n\n"
            "Use the provided context to answer the query. Only use the provided context. "
            'If the answer is not in the context, respond with "I don\'t know".',
        )
    ]
)


def build_rag_pipeline(name: str, embedding_model, chat_model, chunks):
    vectorstore = QdrantVectorStore.from_documents(
        documents=chunks,
        embedding=embedding_model,
        location=":memory:",
        collection_name=f"rag_{name}",
    )
    retriever = vectorstore.as_retriever()

    def run(question: str) -> dict:
        retrieved_docs = retriever.invoke(question)
        context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
        messages = RAG_PROMPT.format_messages(context=context_text, query=question)
        raw_response = chat_model.invoke(messages)
        return {
            "response": raw_response.content,
            "contexts": [doc.page_content for doc in retrieved_docs],
            "usage": getattr(raw_response, "usage_metadata", None) or {},
            "query_embedding_tokens": _tiktoken_len(question),
        }

    return {"name": name, "run": run}


In [10]:
FIREWORKS_BASE_URL = "https://api.fireworks.ai/inference/v1"

fireworks_embeddings = OpenAIEmbeddings(
    model=os.environ.get(
        "FIREWORKS_EMBEDDING_MODEL", "accounts/fireworks/models/qwen3-embedding-8b"
    ),
    openai_api_key=os.environ["FIREWORKS_API_KEY"],
    openai_api_base=FIREWORKS_BASE_URL,
    check_embedding_ctx_length=False,
    dimensions=4096,
)
fireworks_chat = ChatOpenAI(
    model=os.environ.get("FIREWORKS_CHAT_MODEL", "accounts/fireworks/models/gpt-oss-20b"),
    openai_api_key=os.environ["FIREWORKS_API_KEY"],
    openai_api_base=FIREWORKS_BASE_URL,
    temperature=0,
)

fireworks_pipeline = build_rag_pipeline("fireworks", fireworks_embeddings, fireworks_chat, chunks)


In [11]:
openai_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
openai_chat = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

openai_pipeline = build_rag_pipeline("openai", openai_embeddings, openai_chat, chunks)


## 3. Generate a synthetic evaluation set with RAGAS `TestsetGenerator`

Rather than hand-writing Q&A pairs, RAGAS builds a knowledge graph from the source
documents and synthesizes diverse questions + reference answers. We use `gpt-4.1-mini` +
OpenAI embeddings as the (single, shared) generator so the test set itself isn't biased
toward either pipeline under test.

In [12]:
from langchain_core.documents import Document
from ragas.testset import TestsetGenerator

generator_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
generator_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

generator = TestsetGenerator.from_langchain(generator_llm, generator_embeddings)

# ragas builds its own knowledge graph + headline structure from the source documents.
# A couple of PDF pages here are very short (e.g. a section-divider page with ~30 tokens),
# which are below ragas's internal 500-token threshold for headline extraction - but its
# HeadlineSplitter step doesn't apply that same filter, so it crashes on those pages
# ("'headlines' property not found in this node"). Merging all pages into one document
# gives ragas a single, comfortably-sized source to split on its own terms instead.
testset_source_doc = Document(
    page_content="\n\n".join(doc.page_content for doc in documents),
    metadata={"source": DATA_PATH},
)

testset = generator.generate_with_langchain_docs([testset_source_doc], testset_size=10)
testset_df = testset.to_pandas()

testset_df[["user_input", "reference"]]


Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/28 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

...

Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.


Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

,user_input,reference
0,What difference between 2010 feline life stage...,The 2021 AAHA/AAFP Feline Life Stage Guideline...
1,What are the recommended veterinary visit freq...,"Senior cats, defined as those aged over 10 yea..."
2,How should veterinarians approach educating ow...,Veterinarians should educate owners of purebre...
3,What does BCS meen in feline veterinary care?,BCS stands for body condition score in feline ...
4,How important is askin about parasitic disease...,Askin about parasitic disease is very importan...
5,What dietary and hydration strategies have bee...,"In senior cats with early renal disease, feedi..."
6,How should veterinarians address the common he...,Veterinarians should focus on identifying comm...
7,How can multimodal environmental modification ...,Multimodal environmental modification can be b...
8,What role does the 2021 AAFP End of Life Onlin...,The 2021 AAFP End of Life Online Educational T...
9,Considering the unique nutritional and health ...,"Senior cats, defined as those older than 10 ye..."


## 4. Run both pipelines over the generated questions

Each pipeline answers every synthetic question. If a `LANGSMITH_API_KEY` is set, each
provider's run is routed to its own LangSmith project (`rag-fireworks-eval` /
`rag-openai-eval`) so traces, latency, and (where LangSmith recognizes the model) cost
can be inspected and compared side-by-side in the LangSmith UI.

In [13]:
from contextlib import nullcontext

from langchain_core.tracers.context import tracing_v2_enabled
from ragas.dataset_schema import EvaluationDataset, SingleTurnSample


def traced(project_name: str):
    if os.environ.get("LANGSMITH_API_KEY"):
        return tracing_v2_enabled(project_name=project_name)
    return nullcontext()


def run_pipeline_over_testset(pipeline, testset_df):
    samples = []
    usage_records = []
    for _, row in testset_df.iterrows():
        result = pipeline["run"](row["user_input"])
        samples.append(
            SingleTurnSample(
                user_input=row["user_input"],
                response=result["response"],
                retrieved_contexts=result["contexts"],
                reference=row["reference"],
            )
        )
        usage_records.append(
            {
                "provider": pipeline["name"],
                "question": row["user_input"],
                "input_tokens": result["usage"].get("input_tokens", 0),
                "output_tokens": result["usage"].get("output_tokens", 0),
                "query_embedding_tokens": result["query_embedding_tokens"],
            }
        )
    return EvaluationDataset(samples=samples), usage_records


with traced("rag-fireworks-eval"):
    fireworks_dataset, fireworks_usage = run_pipeline_over_testset(fireworks_pipeline, testset_df)

with traced("rag-openai-eval"):
    openai_dataset, openai_usage = run_pipeline_over_testset(openai_pipeline, testset_df)


## 5. Evaluate retrieval quality, faithfulness & end-to-end accuracy

We score both pipelines' outputs with the same judge model (`gpt-4.1-mini`) so the
*judging* is held constant and only the pipeline under test varies:

- **`faithfulness`** - is the answer grounded in the retrieved context? (hallucination check)
- **`answer_relevancy`** - does the answer actually address the question?
- **`context_precision`** - are the retrieved chunks relevant / well-ranked?
- **`context_recall`** - did retrieval surface what was needed to answer correctly?

(`ragas.metrics` emits a `DeprecationWarning` pointing at `ragas.metrics.collections` as
the newer import path in ragas 0.4.x - the classic classes used below are still fully
supported, just silenced here for readability.)

In [14]:
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning, module="ragas")

from ragas import evaluate
from ragas.metrics import AnswerRelevancy, ContextPrecision, ContextRecall, Faithfulness

judge_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
judge_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


def make_metrics():
    return [Faithfulness(), AnswerRelevancy(), ContextPrecision(), ContextRecall()]


fireworks_result = evaluate(
    fireworks_dataset, metrics=make_metrics(), llm=judge_llm, embeddings=judge_embeddings
)
openai_result = evaluate(
    openai_dataset, metrics=make_metrics(), llm=judge_llm, embeddings=judge_embeddings
)


/var/folders/l5/sj4yxxdx2858b09x3g1t44qc0000gn/T/ipykernel_56469/3913128222.py:6: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import AnswerRelevancy, ContextPrecision, ContextRecall, Faithfulness
/var/folders/l5/sj4yxxdx2858b09x3g1t44qc0000gn/T/ipykernel_56469/3913128222.py:6: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import AnswerRelevancy, ContextPrecision, ContextRecall, Faithfulness
/var/folders/l5/sj4yxxdx2858b09x3g1t44qc0000gn/T/ipykernel_56469/3913128222.py:6: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.me

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


In [15]:
import pandas as pd

fireworks_scores = fireworks_result.to_pandas().mean(numeric_only=True)
openai_scores = openai_result.to_pandas().mean(numeric_only=True)

score_comparison = pd.DataFrame(
    {
        "fireworks (gpt-oss-20b)": fireworks_scores,
        "openai (gpt-4.1-mini)": openai_scores,
    }
)
score_comparison


,fireworks (gpt-oss-20b),openai (gpt-4.1-mini)
faithfulness,0.737279,0.941385
answer_relevancy,0.739807,0.908221
context_precision,0.822222,0.933333
context_recall,0.903214,0.953214


## 6. Cost breakdown per provider

LangSmith's built-in cost dashboard prices requests using its known model catalog, which
doesn't include custom Fireworks deployments. So costs here are computed directly from
each response's `usage_metadata` (chat tokens) and `tiktoken` counts (query embedding
tokens - an approximation, since one-time document-indexing embedding cost is a fixed
setup cost independent of query volume and is left out of the per-query figure).

Pricing as of query time (verify against current provider pricing pages before quoting in
your Loom):

- Fireworks `gpt-oss-20b`: $0.07 / $0.30 per 1M input/output tokens ([source](https://docs.fireworks.ai/serverless/pricing))
- Fireworks `qwen3-embedding-8b`: $0.10 per 1M tokens ([source](https://docs.fireworks.ai/serverless/pricing))
- OpenAI `gpt-4.1-mini`: $0.40 / $1.60 per 1M input/output tokens ([source](https://developers.openai.com/api/docs/pricing))
- OpenAI `text-embedding-3-small`: $0.02 per 1M tokens ([source](https://developers.openai.com/api/docs/models/text-embedding-3-small))

In [16]:
PRICING = {
    "fireworks": {
        "chat_input": 0.07 / 1_000_000,
        "chat_output": 0.30 / 1_000_000,
        "embedding": 0.10 / 1_000_000,
    },
    "openai": {
        "chat_input": 0.40 / 1_000_000,
        "chat_output": 1.60 / 1_000_000,
        "embedding": 0.02 / 1_000_000,
    },
}


def summarize_cost(usage_records, prices):
    n = len(usage_records)
    total_input = sum(r["input_tokens"] for r in usage_records)
    total_output = sum(r["output_tokens"] for r in usage_records)
    total_embedding_tokens = sum(r["query_embedding_tokens"] for r in usage_records)

    chat_cost = total_input * prices["chat_input"] + total_output * prices["chat_output"]
    embedding_cost = total_embedding_tokens * prices["embedding"]
    total_cost = chat_cost + embedding_cost

    return {
        "queries": n,
        "chat_input_tokens": total_input,
        "chat_output_tokens": total_output,
        "query_embedding_tokens": total_embedding_tokens,
        "chat_cost_usd": chat_cost,
        "embedding_cost_usd": embedding_cost,
        "total_cost_usd": total_cost,
        "avg_cost_per_query_usd": total_cost / n if n else 0,
    }


fireworks_cost = summarize_cost(fireworks_usage, PRICING["fireworks"])
openai_cost = summarize_cost(openai_usage, PRICING["openai"])

cost_comparison = pd.DataFrame(
    {
        "fireworks (gpt-oss-20b)": fireworks_cost,
        "openai (gpt-4.1-mini)": openai_cost,
    }
)
cost_comparison


,fireworks (gpt-oss-20b),openai (gpt-4.1-mini)
queries,10.000000,10.000000
chat_input_tokens,25221.000000,26519.000000
chat_output_tokens,11668.000000,1895.000000
query_embedding_tokens,275.000000,275.000000
chat_cost_usd,0.005266,0.013640
embedding_cost_usd,0.000028,0.000005
total_cost_usd,0.005293,0.013645
avg_cost_per_query_usd,0.000529,0.001365


In [17]:
print("Projected cost at scale (chat + query-embedding only):\n")
for scale in (1_000, 10_000, 100_000):
    fw = fireworks_cost["avg_cost_per_query_usd"] * scale
    oa = openai_cost["avg_cost_per_query_usd"] * scale
    print(f"{scale:>7,} queries  ->  fireworks: ${fw:>10,.2f}   |   openai: ${oa:>10,.2f}")


Projected cost at scale (chat + query-embedding only):

  1,000 queries  ->  fireworks: $      0.53   |   openai: $      1.36
 10,000 queries  ->  fireworks: $      5.29   |   openai: $     13.65
100,000 queries  ->  fireworks: $     52.93   |   openai: $    136.45


## 7. Wrap-up

For your Loom video, cover:

- **Retrieval/faithfulness/accuracy**: read off `score_comparison` above - which provider
  scored higher on `faithfulness` vs `context_precision`/`context_recall`, and what that
  implies about retrieval quality vs. generation quality for each.
- **Cost at scale**: read off `cost_comparison` and the scaled projections - at what query
  volume (if any) does the cost gap become significant, and does it change your
  provider choice for a production workload?
- **LangSmith**: open the `rag-fireworks-eval` and `rag-openai-eval` projects in the
  LangSmith UI and show the traces/latency side-by-side.